# US App Store keyword opportunity proxy audit

## TL;DR

This notebook recomputes a public-proxy triage index for 15 US iOS search terms. It does not estimate monthly search volume. The strongest niche/intent/solo balance is `before after video`; `video compressor`, `birthday video maker`, `scan old photos`, `add watermark to photos`, and `passport photo` remain testable. Exact Apple Search Popularity is a missing private input.

## Method and limits

Inputs are evidence-bounded 1-5 judgments based on US App Store result relevance, rating accumulation, visible chart signals, pricing/IAP architecture, competitive concentration, product scope and technical constraints observed on 2026-07-18. Ratings are not revenue or keyword volume. The index is a screening aid, not a forecast, probability, v4 opportunity score, or build decision.

In [1]:
import json
from pathlib import Path

DATA_PATH = Path('../data/keyword_candidates.json')
data = json.loads(DATA_PATH.read_text())
rows = data['candidates']

def opportunity_proxy(row):
    return (
        4 * row['relative_volume_proxy']
        + 5 * row['purchase_intent_proxy']
        + 3 * row['solo_fit']
        + 4 * row['wedge_clarity']
        + 4 * (6 - row['competition'])
    )

for row in rows:
    assert opportunity_proxy(row) == row['opportunity_proxy'], row['keyword']

expected_order = sorted(rows, key=lambda row: (-row['opportunity_proxy'], row['rank']))
assert expected_order[0]['keyword'] == 'before after video'
assert all(1 <= row[field] <= 5 for row in rows for field in (
    'relative_volume_proxy', 'purchase_intent_proxy', 'competition', 'solo_fit', 'wedge_clarity'
))
assert data['apple_exact_volume_status'] == 'unavailable_without_apple_ads_account_export'

print('rank | keyword | score | volume | purchase | competition | solo | disposition')
print('--- | --- | --- | --- | --- | --- | --- | ---')
for row in expected_order:
    print(' | '.join(map(str, (
        row['rank'], row['keyword'], row['opportunity_proxy'],
        row['relative_volume_proxy'], row['purchase_intent_proxy'],
        row['competition'], row['solo_fit'], row['disposition']
    ))))

shortlisted = [row['keyword'] for row in rows if row['disposition'] in {'SHORTLIST', 'SHORTLIST_ADJACENT'}]
head_controls = [row['keyword'] for row in rows if row['disposition'] == 'DO_NOT_TARGET_GENERIC']
print('\nPublic-proxy shortlist:', ', '.join(shortlisted))
print('Head-term controls:', ', '.join(head_controls))
print('PASS: formula, bounds, ordering and private-volume limitation assertions')


rank | keyword | score | volume | purchase | competition | solo | disposition
--- | --- | --- | --- | --- | --- | --- | ---
1 | before after video | 84 | 3 | 5 | 2 | 5 | SHORTLIST
2 | birthday video maker | 79 | 3 | 4 | 2 | 5 | SHORTLIST
3 | video compressor | 79 | 4 | 4 | 4 | 5 | SHORTLIST
4 | memorial slideshow | 79 | 2 | 4 | 2 | 5 | QUERY_TEST_ONLY
5 | scan old photos | 76 | 4 | 4 | 4 | 4 | SHORTLIST
6 | add watermark to photos | 75 | 4 | 4 | 4 | 5 | SHORTLIST_ADJACENT
7 | old photo colorizer | 74 | 3 | 5 | 3 | 3 | TECH_ECONOMICS_RAT
8 | passport photo | 73 | 4 | 5 | 5 | 4 | COMPLIANCE_RAT
9 | animate old photos | 71 | 4 | 5 | 4 | 2 | TECH_ECONOMICS_RAT
10 | slideshow maker | 68 | 5 | 5 | 5 | 5 | DO_NOT_TARGET_GENERIC
11 | hidden camera detector | 67 | 4 | 5 | 4 | 2 | TECH_TRUST_RAT_ONLY
12 | photo scanner | 64 | 5 | 4 | 5 | 4 | DO_NOT_TARGET_GENERIC
13 | video to live photo | 63 | 4 | 4 | 5 | 5 | WATCH_NOT_BUILD
14 | background remover | 63 | 5 | 4 | 5 | 5 | DO_NOT_TARGET_GENERIC
1

## Interpretation

- `before after video` has the best public-proxy balance because its job is specific, direct competitors monetize aggressively, and the product can stay on-device.
- `video compressor` is the strongest bounded utility when narrowed to an exact-MB Share Sheet workflow.
- `birthday video maker` and `memorial slideshow` show occasion urgency, but memorial volume needs direct Apple Ads confirmation.
- `animate old photos` and `hidden camera detector` have strong payment signals but fail a solo-feasibility or trust gate until a technical RAT passes.
- Generic `slideshow maker`, `photo scanner`, `background remover`, and `photo restoration` are high-demand controls, not recommended product positions.